## 1. Thiết lập

### Notebook này dùng để làm gì?

Kiểm tra nhanh dữ liệu thô trước khi vẽ biểu đồ, huấn luyện mô hình hay kết luận.

Nếu bảng nguồn bị lỗi thì mọi phần sau đều lệch: đếm sai, join sai, xu hướng giả và giả định sai.

### Các hạng mục được kiểm tra

Năm nhóm kiểm tra thực dụng:
- giá trị thiếu ở các cột khóa
- giá trị âm hoặc bất hợp lệ
- phạm vi số để hiểu bối cảnh kinh doanh
- dòng trống hoặc khóa trùng
- liên kết khóa giữa các bảng


In [ ]:
import duckdb
con = duckdb.connect()

In [ ]:
# load all CSV files into DuckDB
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "dataset").exists() and (candidate / "deliverables").exists():
            return candidate
    raise FileNotFoundError("repo root not found — expected dataset/ and deliverables/.")

ROOT = find_repo_root(Path.cwd())
DATA = ROOT / "dataset"
DELIVERABLES = ROOT / "deliverables"
DELIVERABLES.mkdir(exist_ok=True)

import duckdb

base = DATA
assert base.exists(), f'Data directory not found: {base}'

for f in base.glob("*.csv"):
    con.execute(f"""
     CREATE OR REPLACE TABLE {f.stem}
     AS SELECT * FROM read_csv_auto('{f.as_posix()}')
    """)


In [ ]:
# Quick spot checks after loading the tables
con.execute("SHOW TABLES").df()
con.execute("DESCRIBE orders").df()


,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,order_date,DATE,YES,None,None,None
2,customer_id,BIGINT,YES,None,None,None
3,zip,BIGINT,YES,None,None,None
4,order_status,VARCHAR,YES,None,None,None
5,payment_method,VARCHAR,YES,None,None,None
6,device_type,VARCHAR,YES,None,None,None
7,order_source,VARCHAR,YES,None,None,None


## 2. Giá trị thiếu và dữ liệu bất hợp lệ

Vòng đầu tiên là sanity check cơ bản.

Mục tiêu: tìm chỗ trống ở các cột quan trọng, giá trị âm không nên tồn tại, và các quy tắc cặp dòng phải luôn đúng.

Ví dụ:
- không thiếu id chính
- không có payment hoặc tồn kho âm
- `cogs < price`
- không có doanh thu hay giá vốn âm


In [ ]:
# are any key fields missing?
con.execute("""
select 'products' as table_name,
    sum(product_id is null) as null_primary_key,
    sum(product_name is null) as null_required_field_1,
    sum(price is null) as null_required_field_2
from products

union all
select 'customers',
    sum(customer_id is null),
    sum(zip is null),
    sum(signup_date is null)
from customers

union all
select 'promotions',
    sum(promo_id is null),
    sum(promo_name is null),
    sum(start_date is null)
from promotions

union all
select 'geography',
    sum(zip is null),
    sum(city is null),
    sum(region is null)
from geography

union all
select 'orders',
    sum(order_id is null),
    sum(order_date is null),
    sum(customer_id is null)
from orders

union all
select 'order_items',
    sum(order_id is null),
    sum(product_id is null),
    sum(quantity is null)
from order_items

union all
select 'payments',
    sum(order_id is null),
    sum(payment_method is null),
    sum(payment_value is null)
from payments

union all
select 'shipments',
    sum(order_id is null),
    sum(ship_date is null),
    sum(delivery_date is null)
from shipments

union all
select 'returns',
    sum(return_id is null),
    sum(order_id is null),
    sum(product_id is null)
from returns

union all
select 'reviews',
    sum(review_id is null),
    sum(order_id is null),
    sum(customer_id is null)
from reviews

union all
select 'sales',
    sum("Date" is null),
    sum(revenue is null),
    sum(cogs is null)
from sales

union all
select 'inventory',
    sum(snapshot_date is null),
    sum(product_id is null),
    sum(stock_on_hand is null)
from inventory

union all
select 'web_traffic',
    sum(date is null),
    sum(sessions is null),
    sum(unique_visitors is null)
from web_traffic
""").df()

In [ ]:
# are there negatives or zeros where they should not be?
con.execute("""
select 'products' as table_name,
    sum(price < 0) as negative_value_1,
    sum(cogs < 0) as negative_value_2,
    sum(price = 0) as zero_value_flag
from products

union all
select 'order_items',
    sum(quantity <= 0),
    sum(unit_price < 0),
    sum(discount_amount < 0)
from order_items

union all
select 'payments',
    sum(payment_value < 0),
    sum(installments <= 0),
    sum(payment_value = 0)
from payments

union all
select 'shipments',
    sum(shipping_fee < 0),
    0,
    0
from shipments

union all
select 'returns',
    sum(return_quantity <= 0),
    sum(refund_amount < 0),
    sum(refund_amount = 0)
from returns

union all
select 'inventory',
    sum(stock_on_hand < 0),
    sum(units_received < 0),
    sum(units_sold < 0)
from inventory

union all
select 'web_traffic',
    sum(sessions < 0),
    sum(unique_visitors < 0),
    sum(page_views < 0)
from web_traffic
""").df()

,table_name,negative_value_1,negative_value_2,zero_value_flag
0,products,0.0,0.0,0.0
1,order_items,0.0,0.0,0.0
2,payments,0.0,0.0,0.0
3,shipments,0.0,0.0,0.0
4,returns,0.0,0.0,0.0
5,inventory,0.0,0.0,0.0
6,web_traffic,0.0,0.0,0.0


In [ ]:
# do the values stay inside the rules they should follow?
con.execute("""
-- each product should cost less than its selling price
select 'products' as table_name,
    sum(cogs >= price) as rule_violation_1,
    0 as rule_violation_2,
    0 as rule_violation_3
from products

union all

-- each promotion should have a valid type, a non-negative discount, and an end date after the start date
select 'promotions',
    sum(promo_type not in ('percentage', 'fixed')),
    sum(discount_value < 0),
    sum(end_date < start_date)
from promotions

union all

-- ratings should always be between 1 and 5
select 'reviews',
    sum(rating < 1 or rating > 5),
    0,
    0
from reviews

union all

-- bounce rate is a share between 0 and 1, so it cannot be negative or above 1
select 'web_traffic',
    sum(bounce_rate < 0 or bounce_rate > 1),
    0,
    0
from web_traffic

union all

-- flag any orders with more than 24 installments as unusual
select 'payments',
    sum(installments > 24),
    0,
    0
from payments
""").df()

,table_name,rule_violation_1,rule_violation_2,rule_violation_3
0,products,0.0,0.0,0.0
1,promotions,0.0,0.0,0.0
2,reviews,0.0,0.0,0.0
3,web_traffic,0.0,0.0,0.0
4,payments,0.0,0.0,0.0


In [ ]:
# do related fields in the same row make sense together?
con.execute("""
-- the discount should not be larger than the total value of the line item
select 'order_items' as table_name,
    sum(discount_amount > unit_price * quantity) as logic_issue_1,
    0 as logic_issue_2,
    0 as logic_issue_3
from order_items

union all

-- a refund should come with returned items, and returned items should lead to a refund
select 'returns',
    sum(refund_amount > 0 and return_quantity = 0),
    sum(refund_amount = 0 and return_quantity > 0),
    0
from returns

union all

-- you cannot receive something before it ships, and there is no delivery without a shipment
select 'shipments',
    sum(ship_date is null and delivery_date is not null),
    sum(delivery_date < ship_date),
    sum(delivery_date is null and order_id is not null)
from shipments
""").df()

,table_name,logic_issue_1,logic_issue_2,logic_issue_3
0,order_items,0.0,0.0,0.0
1,returns,0.0,0.0,0.0
2,shipments,0.0,0.0,0.0


## 3. Phạm vi giá trị số

Không phải mọi ngoại lệ đều là lỗi. Dữ liệu kinh doanh đôi khi chỉ đơn giản là nhiều mảnh.

min, max và trung bình giúp thấy quy mô doanh nghiệp và phát hiện điểm cần cảnh giác.


In [ ]:
# check the main numeric columns for values that look unusually low or high
con.execute("""
-- do product prices and costs stay in a reasonable range?
select 'products' as table_name,
    min(price) as min_value_1,
    max(price) as max_value_1,
    avg(price) as avg_value_1,
    min(cogs) as min_value_2,
    max(cogs) as max_value_2,
    avg(cogs) as avg_value_2
from products

union all

-- do order quantities and item-level selling prices look reasonable?
select 'order_items',
    min(quantity),
    max(quantity),
    avg(quantity),
    min(unit_price),
    max(unit_price),
    avg(unit_price)
from order_items

union all

-- do payment amounts and installment counts look reasonable?
select 'payments',
    min(payment_value),
    max(payment_value),
    avg(payment_value),
    min(installments),
    max(installments),
    avg(installments)
from payments

union all

-- do returned quantities and refund amounts stay in a reasonable range?
select 'returns',
    min(return_quantity),
    max(return_quantity),
    avg(return_quantity),
    min(refund_amount),
    max(refund_amount),
    avg(refund_amount)
from returns

union all

-- check the range of daily revenue and cost so we understand the size of the business
select 'sales',
    min(revenue),
    max(revenue),
    avg(revenue),
    min(cogs),
    max(cogs),
    avg(cogs)
from sales
""").df()

,table_name,min_value_1,max_value_1,avg_value_1,min_value_2,max_value_2,avg_value_2
0,products,9.056594,40950.00,4.928216e+03,5.183829,38902.50,3.868347e+03
1,order_items,1.000000,8.00,4.495988e+00,392.570000,43056.00,5.114690e+03
2,payments,389.740000,331570.40,2.423833e+04,1.000000,12.00,3.448319e+00
3,returns,1.000000,8.00,2.743834e+00,458.810000,160937.94,1.278446e+04
4,sales,279813.940000,20905271.35,4.286584e+06,236576.310000,16535857.67,3.695134e+06


## 4. Bản ghi trùng

Một dòng bị lặp lại hai lần có thể phá hỏng phân tích mà rất khó nhận ra.

Số đếm bị phồng lên, join nhân bản dòng và trung bình bị lệch.

Hai kiểu cần kiểm tra:
- id phải duy nhất
- tổ hợp khóa nghiệp vụ không được lặp


In [ ]:
# should the main IDs appear only once?
con.execute("""
-- each main entity should have only one row
select 'products' as table_name,
    count(*) as total_rows,
    count(distinct product_id) as unique_ids,
    count(*) - count(distinct product_id) as duplicate_count
from products

union all

select 'customers',
    count(*),
    count(distinct customer_id),
    count(*) - count(distinct customer_id)
from customers

union all

select 'promotions',
    count(*),
    count(distinct promo_id),
    count(*) - count(distinct promo_id)
from promotions

union all

select 'geography',
    count(*),
    count(distinct zip),
    count(*) - count(distinct zip)
from geography

union all

select 'orders',
    count(*),
    count(distinct order_id),
    count(*) - count(distinct order_id)
from orders

union all

select 'payments',
    count(*),
    count(distinct order_id),
    count(*) - count(distinct order_id)
from payments

union all

select 'returns',
    count(*),
    count(distinct return_id),
    count(*) - count(distinct return_id)
from returns

union all

select 'reviews',
    count(*),
    count(distinct review_id),
    count(*) - count(distinct review_id)
from reviews
""").df()

,table_name,total_rows,unique_ids,duplicate_count
0,products,2412,2412,0
1,customers,121930,121930,0
2,promotions,50,50,0
3,geography,39948,39948,0
4,orders,646945,646945,0
5,payments,646945,646945,0
6,returns,39939,39939,0
7,reviews,113551,113551,0


In [ ]:
# are there repeated combinations that should be unique?
con.execute("""
-- the same product should not appear twice in the same order
select 'order_items' as table_name,
    count(*) as total_rows,
    count(distinct cast(order_id as varchar) || '|' || cast(product_id as varchar)) as unique_combinations,
    count(*) - count(distinct cast(order_id as varchar) || '|' || cast(product_id as varchar)) as duplicate_count
from order_items

union all

-- each product should have only one inventory row per snapshot date
select 'inventory',
    count(*),
    count(distinct cast(snapshot_date as varchar) || '|' || cast(product_id as varchar)),
    count(*) - count(distinct cast(snapshot_date as varchar) || '|' || cast(product_id as varchar))
from inventory

union all

-- each day should appear only once in the sales table
select 'sales',
    count(*),
    count(distinct cast("Date" as varchar)),
    count(*) - count(distinct cast("Date" as varchar))
from sales
""").df()

,table_name,total_rows,unique_combinations,duplicate_count
0,order_items,714669,714653,16
1,inventory,60247,60247,0
2,sales,3833,3833,0


## 5. Liên kết giữa các bảng

Join chỉ đúng khi id thực sự trỏ tới bản ghi tồn tại.

Kiểm tra các mắt xích chính: khách hàng → đơn hàng, đơn hàng → dòng hàng, dòng hàng → sản phẩm, cùng logic cho payments, shipments, returns và reviews.


In [ ]:
# do IDs in one table point to real rows in another table?
con.execute("""
-- every customer zip code should exist in geography
select 'customers_without_geography' as audit_type,
    count(*) as error_count
from customers c
left join geography g
    on c.zip = g.zip
where g.zip is null

union all

-- every order should point to a customer that exists
select 'orders_without_customers',
    count(*)
from orders o
left join customers c
    on o.customer_id = c.customer_id
where c.customer_id is null

union all

-- every shipping zip code should exist in geography
select 'orders_without_geography',
    count(*)
from orders o
left join geography g
    on o.zip = g.zip
where g.zip is null

union all

-- every line item should belong to an order that exists
select 'items_without_orders',
    count(*)
from order_items oi
left join orders o
    on oi.order_id = o.order_id
where o.order_id is null

union all

-- every line item should point to a product that exists
select 'items_without_products',
    count(*)
from order_items oi
left join products p
    on oi.product_id = p.product_id
where p.product_id is null

union all

-- if promo_id is filled in, it should point to a promotion that exists
select 'items_without_promo1',
    count(*)
from order_items oi
left join promotions pr
    on oi.promo_id = pr.promo_id
where oi.promo_id is not null
  and pr.promo_id is null

union all

-- if promo_id_2 is filled in, it should point to a promotion that exists
select 'items_without_promo2',
    count(*)
from order_items oi
left join promotions pr
    on oi.promo_id_2 = pr.promo_id
where oi.promo_id_2 is not null
  and pr.promo_id is null

union all

-- every payment should belong to an order that exists
select 'payments_without_orders',
    count(*)
from payments p
left join orders o
    on p.order_id = o.order_id
where o.order_id is null

union all

-- every shipment should belong to an order that exists
select 'shipments_without_orders',
    count(*)
from shipments s
left join orders o
    on s.order_id = o.order_id
where o.order_id is null

union all

-- every return should belong to an order that exists
select 'returns_without_orders',
    count(*)
from returns r
left join orders o
    on r.order_id = o.order_id
where o.order_id is null

union all

-- every returned product should point to a product that exists
select 'returns_without_products',
    count(*)
from returns r
left join products p
    on r.product_id = p.product_id
where p.product_id is null

union all

-- every review should belong to an order that exists
select 'reviews_without_orders',
    count(*)
from reviews r
left join orders o
    on r.order_id = o.order_id
where o.order_id is null

union all

-- every reviewed product should point to a product that exists
select 'reviews_without_products',
    count(*)
from reviews r
left join products p
    on r.product_id = p.product_id
where p.product_id is null

union all

-- every review should belong to a customer that exists
select 'reviews_without_customers',
    count(*)
from reviews r
left join customers c
    on r.customer_id = c.customer_id
where c.customer_id is null

union all

-- every inventory row should point to a product that exists
select 'inventory_without_products',
    count(*)
from inventory i
left join products p
    on i.product_id = p.product_id
where p.product_id is null
""").df()

,audit_type,error_count
0,customers_without_geography,0
1,orders_without_customers,0
2,orders_without_geography,0
3,items_without_orders,0
4,items_without_products,0
5,items_without_promo1,0
6,items_without_promo2,0
7,payments_without_orders,0
8,shipments_without_orders,0
9,returns_without_orders,0


## 6. Dòng thời gian

Lượt kiểm tra cuối là logic theo thời gian.

Bốn câu hỏi cần xác nhận:
- mỗi bảng có ngày nằm trong khoảng hợp lệ
- `sales` không thiếu ngày lịch
- sự kiện xảy ra theo đúng thứ tự
- giao hàng hoặc trả hàng không diễn ra trước khi đơn được tạo


In [ ]:
# What date range does each date-based table cover?
con.execute("""
-- when did the orders start and end?
select 'orders' as table_name,
    min(order_date) as min_date,
    max(order_date) as max_date
from orders

union all

-- when were promotions first and last active?
select 'promotions',
    min(start_date),
    max(end_date)
from promotions

union all

-- what is the date range of the sales history?
select 'sales',
    min("Date"),
    max("Date")
from sales

union all

-- web traffic starts later than orders and sales; that is expected
select 'web_traffic',
    min(date),
    max(date)
from web_traffic

union all

-- these activity tables should sit inside the main order period
select 'shipments',
    min(ship_date),
    max(ship_date)
from shipments

union all

select 'returns',
    min(return_date),
    max(return_date)
from returns

union all

select 'reviews',
    min(review_date),
    max(review_date)
from reviews

union all

select 'inventory',
    min(snapshot_date),
    max(snapshot_date)
from inventory

order by table_name
""").df()


,table_name,min_date,max_date
0,orders,2012-07-04,2022-12-31
1,sales,2012-07-04,2022-12-31
2,web_traffic,2013-01-01,2022-12-31
3,shipments,2012-07-04,2022-12-29
4,returns,2012-07-11,2022-12-31
5,reviews,2012-07-10,2022-12-31
6,inventory,2012-07-31,2022-12-31


In [ ]:
# Are there any missing days in the sales history?
con.execute("""
with expected as (
    select unnest(
     generate_series(
         (select min("Date") from sales),
         (select max("Date") from sales),
         interval '1 day'
     )
    )::date as expected_date
)
select count(*) as missing_dates
from expected
left join sales
    on expected.expected_date = sales."Date"
where sales."Date" is null
""").df()


,missing_dates
0,0


In [ ]:
# Do the event dates happen in the right order?
con.execute("""
-- orders should come before shipment
select 'shipment_before_order' as check_name,
    count(*) as problem_count
from shipments s
join orders o
    on s.order_id = o.order_id
where s.ship_date < o.order_date

union all

-- delivery should not happen before the order date
select 'delivery_before_order',
    count(*)
from shipments s
join orders o
    on s.order_id = o.order_id
where s.delivery_date < o.order_date

union all

-- returns should not happen before the original order
select 'return_before_order',
    count(*)
from returns r
join orders o
    on r.order_id = o.order_id
where r.return_date < o.order_date

union all

-- returns should happen after delivery
select 'return_before_delivery',
    count(*)
from returns r
join shipments s
    on r.order_id = s.order_id
where r.return_date < s.delivery_date

union all

-- reviews should not happen before the original order
select 'review_before_order',
    count(*)
from reviews rv
join orders o
    on rv.order_id = o.order_id
where rv.review_date < o.order_date

union all

-- reviews should happen after delivery
select 'review_before_delivery',
    count(*)
from reviews rv
join shipments s
    on rv.order_id = s.order_id
where rv.review_date < s.delivery_date

union all

-- if a promotion was used, the order should fall inside that promotion window
select 'promo_1_used_outside_window',
    count(*)
from order_items oi
join orders o
    on oi.order_id = o.order_id
join promotions p
    on oi.promo_id = p.promo_id
where oi.promo_id is not null
  and o.order_date not between p.start_date and p.end_date

union all

-- same check for the second promo field
select 'promo_2_used_outside_window',
    count(*)
from order_items oi
join orders o
    on oi.order_id = o.order_id
join promotions p
    on oi.promo_id_2 = p.promo_id
where oi.promo_id_2 is not null
  and o.order_date not between p.start_date and p.end_date

order by check_name
""").df()


In [ ]:
# Do shipments and returns only appear for the right order statuses?
con.execute("""
-- shipments should exist only for orders that were actually sent out
select 'shipments_bad_status' as check_name,
    count(*) as problem_count
from shipments s
join orders o
    on s.order_id = o.order_id
where o.order_status not in ('shipped', 'delivered', 'returned')

union all

-- returns should exist only for orders marked as returned
select 'returns_bad_status',
    count(*)
from returns r
join orders o
    on r.order_id = o.order_id
where o.order_status <> 'returned'
""").df()


,check_name,violation_count
0,shipments_bad_status,0
1,returns_bad_status,0
